#### Question 3: MLP for classification of latent space.

In [1]:
import os
import tqdm
import torch
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
from torchvision.utils import save_image
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
# Data loading
path = os.path.join(os.getcwd(), 'data','butterfly')
MEAN = torch.tensor([0.47998455, 0.46537864, 0.33653912]) # After Calculation, Mean is directly given to the code.
STD = torch.tensor([0.22608626, 0.22103393, 0.21348171])

image_size = 128
batch_size = 1
workers = 12
n_channels = 3
latent_dim = 256 
lr = 0.0002
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

train_data = ImageFolder(path, transform=transform)
loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=workers)

classes = train_data.classes

In [3]:
class Encoder(nn.Module):
    def __init__(self):
        super(Encoder, self).__init__()
        self.latent_dim = latent_dim

        self.input_dim = n_channels
        self.output_dim = latent_dim  
        self.encode = nn.Sequential(
            nn.Conv2d(self.input_dim, 8, 4, 2, 1, bias=False), # 64 x 64
            nn.BatchNorm2d(8),
            nn.LeakyReLU(0.2),

            nn.Conv2d(8, 16, 4, 2, 1, bias=False), # 32 x 32
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.2),

            nn.Conv2d(16, 32, 4, 2, 1, bias=False), # 16 x 16
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),

            nn.Conv2d(32, 64, 4, 2, 1, bias=False), # 8 x 8
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1, bias=False), # 4 x 4
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Conv2d(128, 256, 4, 2, 1, bias=False), # 2 x 2
            nn.BatchNorm2d(256),
            

        )
        self.layer_mean = nn.Linear(256*2*2, self.output_dim)
        self.layer_var = nn.Linear(256*2*2, self.output_dim)
    def forward(self, x):
        x = self.encode(x)
        x = torch.flatten(x, start_dim=1)
        mean = self.layer_mean(x)
        var = self.layer_var(x)
        return mean, var
    
class Decoder(nn.Module):
    def __init__(self):
        super(Decoder, self).__init__()
        self.batch_size = batch_size
        self.latent_dim = latent_dim

        self.input_dim = latent_dim
        self.output_dim = n_channels

        self.fc = nn.Linear(self.input_dim, self.input_dim*2*2)
        
        self.decode = nn.Sequential(

            nn.ConvTranspose2d(self.input_dim, 128, 4, 2, 1, bias=False),  #  4 x 4
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),  #  8 x 8
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False),    # 16 x 16
            nn.BatchNorm2d(32),
            nn.ReLU(True),

            nn.ConvTranspose2d(32, 16, 4, 2, 1, bias=False),    # 32 x 32
            nn.BatchNorm2d(16),
            nn.ReLU(True),

            nn.ConvTranspose2d(16, 8, 4, 2, 1, bias=False),     # 64 x 64
            nn.BatchNorm2d(8),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(8, self.output_dim, 4, 2, 1, bias=False), # 128 x 128
            
            nn.Tanh()
        )
    def forward(self, x):
        x = self.fc(x)
        x = x.view(x.size(0), self.latent_dim,2,2)
        out = self.decode(x)
        return (out + 1) / 2
    
class Vanilla_VAE(nn.Module):
    def __init__(self):
        super(Vanilla_VAE, self).__init__()
        self.latent_dim = latent_dim
        self.encoder = Encoder()
        self.decoder = Decoder()

    def reparameterization(self, mean, var):
        std = torch.exp(0.5*var) # standard deviation from log(variance)
        epsilon = torch.randn_like(std) # sampling from normal (0,1)
        return mean + std * epsilon
    
    def forward(self, x):
        mean, log_var = self.encoder(x)
        z = self.reparameterization(mean, log_var)
        x_hat = self.decoder(z) # reconstructed image
        return x_hat, mean, log_var
            
    def sample(self, z):
        return self.decoder(z)
    
def weights_init(c):
    class_name = c.__class__.__name__
    if class_name.find("Conv") != -1:
        nn.init.normal_(c.weight.data, 0.0, 0.02)
        # mean of Conv layer weights is taken 0.0
    elif class_name.find("BatchNorm") != -1:
        nn.init.normal_(c.weight.data, 1.0, 0.0)
        # mean of Batch Normalization layer weights is taken 0.0
        nn.init.constant_(c.bias.data, 0)
def read_epoch_count(file_path):
    with open(file_path, 'r') as file:
        return int(file.readline().strip())


In [4]:
vae = torch.load(f'new_vanilla_vae.pth',weights_only = False).to(device)
vae.eval()
latent = []
for image, label in loader:
    datapoint = []
    datapoint.append(vae.encoder(image.to(device))[0].detach().cpu())
    datapoint.append(label)
    latent.append(datapoint)    


In [5]:
X, y = zip(*latent)
X = torch.cat(X)
y = torch.cat(y)
print(X.shape,y.shape)

#Training data
dataset = torch.utils.data.TensorDataset(X, y)
train_length = int(0.8*len(dataset))
val_length = int(0.1*len(dataset))
test_length = int(len(dataset) - train_length - val_length)
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_length, val_length, test_length])
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=256, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=256, shuffle=True)

torch.Size([6499, 256]) torch.Size([6499])


In [19]:
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=512):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim*2),
            nn.ReLU(),
            nn.Linear(hidden_dim*2, hidden_dim*4),
            nn.ReLU(),
            nn.Linear(hidden_dim*4, output_dim),
        )
    def forward(self, x):
        return self.model(x)
    
def train_model(model, train_loader, val_loader, epochs, lr, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        for data in train_loader:
            x, y = data
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            y_hat = model(x)
            loss = criterion(y_hat, y)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            correct = 0
            total = 0
            for data in val_loader:
                x, y = data
                x = x.to(device)
                y = y.to(device)
                y_hat = model(x)
                _, predicted = torch.max(y_hat.data, 1)
                total += y.size(0)
                correct += (predicted == y).sum().item()
            accuracy = correct / total
    print(f'Epoch: {epoch+1}/{epochs}, Loss: {loss.item()}, Accuracy: {accuracy*100}%')
    return model

model = MLP(256, len(classes)).to(device)
model.apply(weights_init)
model = train_model(model, train_loader, val_loader, 100, 0.0002, device)
torch.save(model, 'output/vanilla_vae_mlp.pth')



Epoch: 100/100, Loss: 0.05273602157831192, Accuracy: 32.20338983050847%


By MLP classification of latent vectors with dimension 256, we get classfication accuracy around 32-37%
#### Observation:
1) Classification is faster in latent space instead of classification of image directly through CNNs.
2) If latent dimension size is not sufficient to capture all the details in the image than we can not obtain good accuracy of classification like CNN-based classifier.

| Latent Dimension | Hidden Dim = 512 | Hidden Dim = 1024 | Hidden Dim = 2048 | Observations |
|------------------|---------------------------------|----------------------------------|---------------------------------|--------------|
| 128               |     13.713%                       | 12.326%                           | 12.4807%                      | Lower dimensional latent space may limit representation capacity, affecting accuracy. |
| 256               | 21.263%                         |26.964%                           | 25.420  %                      | Improved accuracy with larger network size, but increased computational cost. |
<!-- | 512              | 13.097%                           | 13.097%                           | 12.018%                            | High dimensional latent space should capture more information and give high accuracy the model is not trained properly. so we are getting low accuracy | -->

